# BigNeuron Benchmark — Evaluation

| Method | Description |
|--------|-------------|
| **ours** | Riemannian FMM tracer (per-Z soma detection) |
| **vaa3d** | Vaa3D APP2 algorithm |

**Metrics (via pyneval)**
- `ssd` — Spatial Structure Distance (낮을수록 좋음)
- `recall / precision / f1` — branch overlap

> gold standard SWC는 원본 픽셀 단위 → µm 변환 후 비교

In [ ]:
from pathlib import Path
import re, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
import tifffile
import warnings
warnings.filterwarnings('ignore')

ROOT        = Path('..').resolve()
GOLD_DIR    = ROOT / 'data' / 'gold_standard'
RESULTS_DIR = ROOT / 'results'
OUT_DIR     = ROOT / 'methods' / 'ours' / 'output'

# pyneval 실행 파일 경로 (tracing conda env)
PYNEVAL = '/opt/homebrew/Caskroom/miniforge/base/envs/tracing/bin/pyneval'

METHODS = ['ours', 'vaa3d']
METHOD_COLORS = {'ours': '#4c9be8', 'vaa3d': '#e87b4c', 'gold': '#f5c518'}

# 결과가 있는 샘플만 자동 수집
samples = sorted(set(
    p.stem for p in (RESULTS_DIR / 'ours').glob('*.swc')
) & set(
    p.stem for p in GOLD_DIR.glob('*.swc')
))
print(f'샘플: {samples}')

## Helper functions

In [ ]:
def gold_voxel(stem):
    """pixelsize.txt에서 (voxel_xy, voxel_z) 읽기 (µm)"""
    txt = (GOLD_DIR / f'{stem}.pixelsize.txt').read_text()
    m = re.search(r'Voxel size:\s*([\d.]+)x[\d.]+x([\d.]+)', txt, re.I)
    return float(m.group(1)), float(m.group(2))


def gold_to_um_file(stem):
    """gold standard SWC 픽셀 좌표 → µm 변환, 임시 파일 반환"""
    vxy, vz = gold_voxel(stem)
    lines = []
    for line in (GOLD_DIR / f'{stem}.swc').read_text().splitlines():
        if line.startswith('#') or not line.strip():
            lines.append(line); continue
        p = line.split()
        p[2] = f'{float(p[2])*vxy:.4f}'
        p[3] = f'{float(p[3])*vxy:.4f}'
        p[4] = f'{float(p[4])*vz:.4f}'
        lines.append(' '.join(p))
    tmp = Path(f'/tmp/{stem}_gold_um.swc')
    tmp.write_text('\n'.join(lines))
    return tmp


def pyneval_score(test_swc, gold_um_swc):
    """pyneval 실행 → dict 반환"""
    r = subprocess.run(
        [PYNEVAL, '--test', str(test_swc), '--gold', str(gold_um_swc), '--metric', 'ssd'],
        capture_output=True, text=True)
    out = {}
    for line in r.stdout.splitlines():
        if '=' in line:
            k, _, v = line.partition('=')
            try: out[k.strip()] = float(v.strip())
            except: pass
    return out


def load_swc_segments(swc_path, scale_xy=1.0, scale_z=1.0):
    """
    SWC 읽어서 세그먼트 목록 반환.
    scale_xy, scale_z: 픽셀→µm 변환 (gold standard용). ours는 1.0.
    반환: nodes dict {nid: (x_um, y_um, z_um)}, segs list [(p1,p2)]
    """
    nodes = {}
    for line in Path(swc_path).read_text().splitlines():
        if line.startswith('#') or not line.strip(): continue
        p = line.split()
        if len(p) < 7: continue
        nodes[int(p[0])] = (float(p[2])*scale_xy,
                             float(p[3])*scale_xy,
                             float(p[4])*scale_z,
                             int(p[6]))
    segs = []
    for nid, (x, y, z, pid) in nodes.items():
        if pid == -1 or pid not in nodes: continue
        px, py, pz, _ = nodes[pid]
        segs.append(((x, y, z), (px, py, pz)))
    return segs

## 정량 지표 계산

In [ ]:
rows = []
for stem in samples:
    gold_um = gold_to_um_file(stem)
    for method in METHODS:
        test = RESULTS_DIR / method / f'{stem}.swc'
        if not test.exists(): continue
        scores = pyneval_score(test, gold_um)
        rows.append({
            'sample': stem, 'method': method,
            'ssd':       scores.get('ssd_score',       float('nan')),
            'recall':    scores.get('recall',           float('nan')),
            'precision': scores.get('precision',        float('nan')),
            'f1':        scores.get('f1_score',         float('nan')),
        })
        print(f"{method:8s}  {stem}  ssd={scores.get('ssd_score', float('nan')):.2f}  "
              f"f1={scores.get('f1_score', float('nan')):.3f}")

df = pd.DataFrame(rows)
df.to_csv('results_raw.csv', index=False)
df

## 요약 테이블

In [ ]:
METRICS = ['ssd', 'recall', 'precision', 'f1']
summary = df.groupby('method')[METRICS].mean().round(4)
summary.style.highlight_min(subset=['ssd'], color='lightgreen') \
             .highlight_max(subset=['recall','precision','f1'], color='lightgreen')

## 지표 비교 그래프

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
palette = [METHOD_COLORS[m] for m in METHODS if m in METHOD_COLORS]

for ax, metric in zip(axes, METRICS):
    vals = [df[df.method == m][metric].values for m in METHODS]
    bars = ax.bar(METHODS, [np.nanmean(v) for v in vals], color=palette, alpha=0.85)
    for bar, vs in zip(bars, vals):
        for v in vs:
            if not np.isnan(v):
                ax.scatter(bar.get_x() + bar.get_width()/2, v,
                           color='black', s=30, zorder=5)
    ax.set_title(metric)
    ax.set_ylim(0, None)

axes[0].set_ylabel('score')
plt.suptitle('BigNeuron Benchmark', fontsize=13)
plt.tight_layout()
plt.savefig('benchmark_results.png', dpi=150, bbox_inches='tight')
plt.show()

## MIP 오버레이 시각화

각 샘플에 대해 XY / XZ / YZ 3방향 MIP 위에 SWC를 겹쳐 표시합니다.
- **이미지**: 전처리된 스택 (stack_preprocessed.tif)
- **Gold standard**: 노란색
- **Ours**: 파란색
- **Vaa3D**: 주황색

In [ ]:
def segs_to_lc_xy(segs, voxel_iso, color, lw=0.7, alpha=0.9):
    """XY 투영 (Z방향 MIP): SWC µm → (col=x/v, row=y/v)"""
    lines = [[(x1/voxel_iso, y1/voxel_iso),
               (x2/voxel_iso, y2/voxel_iso)] for (x1,y1,z1),(x2,y2,z2) in segs]
    return LineCollection(lines, colors=color, linewidths=lw, alpha=alpha)

def segs_to_lc_xz(segs, voxel_iso, color, lw=0.7, alpha=0.9):
    """XZ 투영 (Y방향 MIP): col=x/v, row=z/v"""
    lines = [[(x1/voxel_iso, z1/voxel_iso),
               (x2/voxel_iso, z2/voxel_iso)] for (x1,y1,z1),(x2,y2,z2) in segs]
    return LineCollection(lines, colors=color, linewidths=lw, alpha=alpha)

def segs_to_lc_yz(segs, voxel_iso, color, lw=0.7, alpha=0.9):
    """YZ 투영 (X방향 MIP): col=y/v, row=z/v"""
    lines = [[(y1/voxel_iso, z1/voxel_iso),
               (y2/voxel_iso, z2/voxel_iso)] for (x1,y1,z1),(x2,y2,z2) in segs]
    return LineCollection(lines, colors=color, linewidths=lw, alpha=alpha)


def mip_figure(stem):
    stack_path = OUT_DIR / stem / 'stack_preprocessed.tif'
    if not stack_path.exists():
        print(f'{stem}: stack_preprocessed.tif 없음'); return

    # ── 이미지 로드 & MIP ──────────────────────────────────────────
    stack = tifffile.imread(str(stack_path)).astype(np.float32)
    voxel_iso = float(np.load(str(OUT_DIR / stem / 'preprocess_meta.npz'))['voxel_iso'])
    mip_xy = stack.max(axis=0)   # (NY, NX)
    mip_xz = stack.max(axis=1)   # (NZ, NX)
    mip_yz = stack.max(axis=2)   # (NZ, NY)

    # ── SWC 로드 ────────────────────────────────────────────────────
    vxy, vz = gold_voxel(stem)
    swc_info = [
        ('Gold',  GOLD_DIR / f'{stem}.swc',      METHOD_COLORS['gold'],   vxy, vz),
        ('Ours',  RESULTS_DIR / 'ours' / f'{stem}.swc',  METHOD_COLORS['ours'],  1.0, 1.0),
        ('Vaa3D', RESULTS_DIR / 'vaa3d' / f'{stem}.swc', METHOD_COLORS['vaa3d'], 1.0, 1.0),
    ]
    swc_segs = []
    for label, path, color, sxy, sz in swc_info:
        if not path.exists():
            swc_segs.append((label, None, color)); continue
        segs = load_swc_segments(path, scale_xy=sxy, scale_z=sz)
        swc_segs.append((label, segs, color))

    # ── 그리기 ──────────────────────────────────────────────────────
    nrows = 1 + len(swc_segs)  # image row + method rows
    fig, axes = plt.subplots(nrows, 3, figsize=(15, 4*nrows))
    proj_titles = ['XY (Z-MIP)', 'XZ (Y-MIP)', 'YZ (X-MIP)']
    mips        = [mip_xy,       mip_xz,        mip_yz]
    lc_fns      = [segs_to_lc_xy, segs_to_lc_xz, segs_to_lc_yz]

    # 이미지 전용 row
    for col, (mip, title) in enumerate(zip(mips, proj_titles)):
        ax = axes[0, col]
        ax.imshow(mip, cmap='gray', origin='upper',
                  aspect='auto' if col > 0 else 'equal',
                  vmax=np.percentile(mip[mip>0], 99) if mip.max() > 0 else 1)
        ax.set_title(title, fontsize=11)
        if col == 0: ax.set_ylabel('Image', fontsize=11)
        ax.axis('off')

    # 각 방법 overlay row
    for row, (label, segs, color) in enumerate(swc_segs, start=1):
        for col, (mip, lc_fn) in enumerate(zip(mips, lc_fns)):
            ax = axes[row, col]
            ax.imshow(mip, cmap='gray', origin='upper',
                      aspect='auto' if col > 0 else 'equal',
                      vmax=np.percentile(mip[mip>0], 99) if mip.max() > 0 else 1)
            if segs:
                ax.add_collection(lc_fn(segs, voxel_iso, color))
            if col == 0: ax.set_ylabel(label, fontsize=11, color=color)
            ax.axis('off')

    plt.suptitle(f'{stem}  —  MIP + SWC overlay  (voxel={voxel_iso:.4f} µm)',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(f'mip_{stem}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: mip_{stem}.png')

In [ ]:
for stem in samples:
    mip_figure(stem)